In [ ]:
%pip -q install -U openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 563.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 8.5 MB/s eta 0:00:00


In [ ]:
import json
import os
from copy import deepcopy
from getpass import getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

client = OpenAI()
MODEL = "gpt-5.6-luna"
MAX_RODADAS = 8
MAX_OUTPUT_TOKENS = 350
LIMITE_ESTIMADO_USD = 0.02
PRECO_INPUT_1M = 0.20
PRECO_OUTPUT_1M = 1.20

OPENAI_API_KEY: ··········


In [ ]:
CENARIO_ORIGINAL = {
    "metricas": {
        "api-gateway": {
            "cpu_pct": 38, "memoria_pct": 61, "latencia_p95_ms": 6200,
            "taxa_erro_pct": 18.4, "instancias": 3
        },
        "recommendation-api": {
            "cpu_pct": 98, "memoria_pct": 84, "latencia_p95_ms": 5870,
            "taxa_erro_pct": 31.7, "instancias": 48
        },
        "catalog-api": {
            "cpu_pct": 42, "memoria_pct": 55, "latencia_p95_ms": 93,
            "taxa_erro_pct": 0.2, "instancias": 4
        },
        "checkout-api": {
            "cpu_pct": 47, "memoria_pct": 58, "latencia_p95_ms": 310,
            "taxa_erro_pct": 0.6, "instancias": 5
        },
    },
    "logs": {
        "api-gateway": [
            "14:03:11 WARN upstream recommendation-api timeout after 5000ms",
            "14:03:14 WARN upstream recommendation-api timeout after 5000ms",
            "14:03:18 ERROR circuit breaker recommendation-api OPEN",
        ],
        "recommendation-api": [
            "14:02:57 WARN catalog response missing field related_items",
            "14:02:57 INFO invoking fallback target=recommendation-api depth=1",
            "14:02:58 INFO invoking fallback target=recommendation-api depth=8",
            "14:02:59 ERROR maximum fallback depth reached depth=25",
            "14:03:00 WARN request queue saturation=96%",
        ],
        "catalog-api": [
            "14:02:55 INFO GET /items/842 200 71ms",
            "14:03:01 INFO GET /items/163 200 66ms",
        ],
        "checkout-api": [
            "14:03:02 INFO POST /checkout 200 288ms",
            "14:03:07 INFO POST /checkout 200 302ms",
        ],
    },
    "deploys": {
        "api-gateway": [{"id": "dep-410", "versao": "5.8.1", "hora": "ontem 16:20", "status": "concluido"}],
        "recommendation-api": [
            {
                "id": "dep-442", "versao": "2.4.0", "hora": "13:58", "status": "concluido",
                "mudancas": [
                    "novo fallback para resposta incompleta do catalogo",
                    "FALLBACK_SERVICE: catalog-api -> recommendation-api",
                ],
            },
            {"id": "dep-431", "versao": "2.3.7", "hora": "ontem 11:40", "status": "concluido"},
        ],
        "catalog-api": [{"id": "dep-405", "versao": "3.1.2", "hora": "3 dias atras", "status": "concluido"}],
        "checkout-api": [{"id": "dep-398", "versao": "7.2.0", "hora": "5 dias atras", "status": "concluido"}],
    },
    "configuracoes": {
        "api-gateway": {"timeout_ms": 5000, "retries": 1},
        "recommendation-api": {
            "fallback_service": "recommendation-api",
            "fallback_max_depth": 25,
            "autoscaling_min": 2,
            "autoscaling_max": 50,
            "autoscaling_cpu_target_pct": 60,
        },
        "catalog-api": {"cache_ttl_s": 300, "replicas": 4},
        "checkout-api": {"replicas": 5, "timeout_ms": 1500},
    },
    "custos": {
        "api-gateway": {"baseline_usd_h": 0.22, "atual_usd_h": 0.24},
        "recommendation-api": {"baseline_usd_h": 0.31, "atual_usd_h": 7.44},
        "catalog-api": {"baseline_usd_h": 0.38, "atual_usd_h": 0.40},
        "checkout-api": {"baseline_usd_h": 0.51, "atual_usd_h": 0.52},
    },
    "eventos": [
        {"hora": "13:58", "tipo": "deploy", "servico": "recommendation-api", "id": "dep-442"},
        {"hora": "14:01", "tipo": "autoscaling", "servico": "recommendation-api", "detalhe": "2 -> 18 instancias"},
        {"hora": "14:03", "tipo": "autoscaling", "servico": "recommendation-api", "detalhe": "18 -> 48 instancias"},
        {"hora": "14:03", "tipo": "alerta", "servico": "api-gateway", "detalhe": "latencia p95 acima de 5s"},
    ],
}

ambiente = deepcopy(CENARIO_ORIGINAL)
print("Serviços monitorados:", ", ".join(ambiente["metricas"].keys()))


Serviços monitorados: api-gateway, recommendation-api, catalog-api, checkout-api


In [ ]:
def _validar_servico(servico):
    if servico not in ambiente["metricas"]:
        return {"erro": f"Servico desconhecido: {servico}", "disponiveis": list(ambiente["metricas"])}
    return None


def consultar_metricas(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, **ambiente["metricas"][servico]}


def buscar_logs(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "linhas": ambiente["logs"].get(servico, [])}


def consultar_deploys(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "deploys": ambiente["deploys"].get(servico, [])}


def consultar_configuracao(servico):
    erro = _validar_servico(servico)
    return erro or {"servico": servico, "configuracao": ambiente["configuracoes"].get(servico, {})}


def consultar_custos(servico):
    erro = _validar_servico(servico)
    if erro:
        return erro
    custo = ambiente["custos"][servico]
    multiplicador = round(custo["atual_usd_h"] / custo["baseline_usd_h"], 1)
    return {"servico": servico, **custo, "multiplicador": multiplicador}


def consultar_eventos_recentes():
    return {"eventos": ambiente["eventos"]}


def solicitar_rollback(deploy_id, justificativa):
    # Esta ferramenta NÃO executa o rollback. Ela abre uma solicitação.
    encontrado = None
    for servico, deploys in ambiente["deploys"].items():
        for deploy in deploys:
            if deploy["id"] == deploy_id:
                encontrado = {"servico": servico, **deploy}
                break
    if not encontrado:
        return {"status": "rejeitado", "motivo": "deploy inexistente"}
    return {
        "status": "AGUARDANDO_APROVACAO_HUMANA",
        "deploy": encontrado,
        "justificativa_registrada": justificativa,
        "executado": False,
    }


In [ ]:
print(json.dumps(consultar_metricas("api-gateway"), indent=2, ensure_ascii=False))
print(json.dumps(consultar_custos("recommendation-api"), indent=2, ensure_ascii=False))


{
  "servico": "api-gateway",
  "cpu_pct": 38,
  "memoria_pct": 61,
  "latencia_p95_ms": 6200,
  "taxa_erro_pct": 18.4,
  "instancias": 3
}
{
  "servico": "recommendation-api",
  "baseline_usd_h": 0.31,
  "atual_usd_h": 7.44,
  "multiplicador": 24.0
}


In [ ]:
def schema_servico(nome, descricao):
    return {
        "type": "function",
        "name": nome,
        "description": descricao,
        "parameters": {
            "type": "object",
            "properties": {
                "servico": {
                    "type": "string",
                    "enum": ["api-gateway", "recommendation-api", "catalog-api", "checkout-api"],
                    "description": "Nome do servico",
                }
            },
            "required": ["servico"],
            "additionalProperties": False,
        },
        "strict": True,
    }


TOOLS = [
    schema_servico("consultar_metricas", "Consulta CPU, memoria, latencia, erros e numero de instancias de um servico."),
    schema_servico("buscar_logs", "Busca as linhas recentes de log de um servico. Use para investigar sintomas e erros."),
    schema_servico("consultar_deploys", "Lista deploys recentes e mudancas conhecidas de um servico."),
    schema_servico("consultar_configuracao", "Consulta a configuracao operacional atual de um servico."),
    schema_servico("consultar_custos", "Compara o custo atual por hora de um servico com seu baseline."),
    {
        "type": "function",
        "name": "consultar_eventos_recentes",
        "description": "Lista eventos cloud recentes em ordem temporal: deploys, autoscaling e alertas.",
        "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False},
        "strict": True,
    },
    {
        "type": "function",
        "name": "solicitar_rollback",
        "description": "Solicita rollback de um deploy suspeito. Apenas abre pedido; exige aprovacao humana para executar.",
        "parameters": {
            "type": "object",
            "properties": {
                "deploy_id": {"type": "string", "description": "ID exato de um deploy consultado"},
                "justificativa": {"type": "string", "description": "Evidencia objetiva que justifica o rollback"},
            },
            "required": ["deploy_id", "justificativa"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

print(f"{len(TOOLS)} ferramentas declaradas para o modelo.")


7 ferramentas declaradas para o modelo.


In [ ]:
# dispatcher
FUNCOES_PERMITIDAS = {
    "consultar_metricas": consultar_metricas,
    "buscar_logs": buscar_logs,
    "consultar_deploys": consultar_deploys,
    "consultar_configuracao": consultar_configuracao,
    "consultar_custos": consultar_custos,
    "consultar_eventos_recentes": consultar_eventos_recentes,
    "solicitar_rollback": solicitar_rollback,
}

def executar_ferramenta(nome, argumentos):
    if nome not in FUNCOES_PERMITIDAS:
        return {"erro": f"Ferramenta nao permitida: {nome}"}
    try:
        return FUNCOES_PERMITIDAS[nome](**argumentos)
    except Exception as exc:
        return {"erro": type(exc).__name__, "detalhe": str(exc)}


In [ ]:
INSTRUCOES = '''
Voce e o agente legista de incidentes Cloud da Loja Nebulosa.
Seu objetivo e investigar a causa raiz usando apenas evidencias retornadas pelas ferramentas.

Regras:
- Nao trate proximidade temporal como prova de causalidade.
- Investigue primeiro; nao chute valores ou eventos.
- Faca consultas curtas e seletivas.
- Diferencie sintoma, causa raiz e impacto financeiro.
- Se houver evidencia suficiente, solicite rollback com justificativa objetiva.
- Nunca diga que uma acao foi executada quando a ferramenta disser que aguarda aprovacao.
- Encerre com: diagnostico, evidencias, impacto, acao recomendada e nivel de confianca.
- Seja conciso.
'''.strip()

INCIDENTE = '''
INC-2026-0911: desde 14h03, usuarios relatam lentidao no e-commerce.
O api-gateway disparou alerta de latencia e o custo cloud esta subindo.
Descubra a causa raiz e recomende a resposta operacional mais segura.
'''.strip()


In [ ]:
def estimar_custo(input_tokens, output_tokens):
    return (
        input_tokens * PRECO_INPUT_1M / 1_000_000
        + output_tokens * PRECO_OUTPUT_1M / 1_000_000
    )


def investigar_incidente(incidente=INCIDENTE):
    historico = [{"role": "user", "content": incidente}]
    total_input = 0
    total_output = 0

    for rodada in range(1, MAX_RODADAS + 1):
        resposta = client.responses.create(
            model=MODEL,
            instructions=INSTRUCOES,
            input=historico,
            tools=TOOLS,
            parallel_tool_calls=False,
            reasoning={"effort": "none"},
            max_output_tokens=MAX_OUTPUT_TOKENS,
            store=False,
        )

        if resposta.usage:
            total_input += resposta.usage.input_tokens
            total_output += resposta.usage.output_tokens

        custo = estimar_custo(total_input, total_output)
        print(f"\n--- Rodada {rodada} | custo estimado acumulado: US$ {custo:.6f} ---")

        # Preserva mensagens e chamadas do modelo para a proxima rodada.
        historico += resposta.output
        chamadas = [item for item in resposta.output if item.type == "function_call"]

        if not chamadas:
            print("\nRELATORIO FINAL\n")
            print(resposta.output_text)
            return {
                "resposta": resposta.output_text,
                "input_tokens": total_input,
                "output_tokens": total_output,
                "custo_estimado_usd": custo,
                "rodadas": rodada,
            }

        for chamada in chamadas:
            argumentos = json.loads(chamada.arguments)
            print(f"Agente pediu: {chamada.name}({argumentos})")
            resultado = executar_ferramenta(chamada.name, argumentos)
            print("Observacao:", json.dumps(resultado, ensure_ascii=False))
            historico.append({
                "type": "function_call_output",
                "call_id": chamada.call_id,
                "output": json.dumps(resultado, ensure_ascii=False),
            })

        if custo >= LIMITE_ESTIMADO_USD:
            print(f"\nLimite didatico estimado de US$ {LIMITE_ESTIMADO_USD:.2f} atingido.")
            return {
                "resposta": None,
                "input_tokens": total_input,
                "output_tokens": total_output,
                "custo_estimado_usd": custo,
                "rodadas": rodada,
                "interrompido": True,
            }

    print("\nNumero maximo de rodadas atingido. O agente foi interrompido pelo software.")
    return {
        "resposta": None,
        "input_tokens": total_input,
        "output_tokens": total_output,
        "custo_estimado_usd": estimar_custo(total_input, total_output),
        "rodadas": MAX_RODADAS,
        "interrompido": True,
    }


In [ ]:
resultado = investigar_incidente()
resultado


--- Rodada 1 | custo estimado acumulado: US$ 0.000163 ---
Agente pediu: consultar_metricas({'servico': 'api-gateway'})
Observacao: {"servico": "api-gateway", "cpu_pct": 38, "memoria_pct": 61, "latencia_p95_ms": 6200, "taxa_erro_pct": 18.4, "instancias": 3}

--- Rodada 2 | custo estimado acumulado: US$ 0.000342 ---
Agente pediu: buscar_logs({'servico': 'api-gateway'})
Observacao: {"servico": "api-gateway", "linhas": ["14:03:11 WARN upstream recommendation-api timeout after 5000ms", "14:03:14 WARN upstream recommendation-api timeout after 5000ms", "14:03:18 ERROR circuit breaker recommendation-api OPEN"]}

--- Rodada 3 | custo estimado acumulado: US$ 0.000541 ---
Agente pediu: consultar_metricas({'servico': 'recommendation-api'})
Observacao: {"servico": "recommendation-api", "cpu_pct": 98, "memoria_pct": 84, "latencia_p95_ms": 5870, "taxa_erro_pct": 31.7, "instancias": 48}

--- Rodada 4 | custo estimado acumulado: US$ 0.000760 ---
Agente pediu: consultar_deploys({'servico': 'recommendat

{'resposta': '**Diagnóstico:** provável recursão do fallback introduzido no `recommendation-api` versão 2.4.0, causando saturação e propagação de latência ao `api-gateway`.\n\n**Evidências:**\n- Deploy `dep-442` concluído às 13:58, com novo fallback para `recommendation-api`.\n- Configuração atual: `fallback_service=recommendation-api` e `fallback_max_depth=25`.\n- Desde 14:03, o gateway registra timeouts de 5 s e abertura do circuit breaker para esse serviço.\n- `recommendation-api`: CPU 98%, latência p95 de 5.870 ms e erro de 31,7%.\n- Escalonamento para 48 instâncias, próximo do máximo de 50.\n- Custo: US$ 7,44/h contra baseline de US$ 0,31/h, aumento de 24 vezes.\n\n**Impacto:** lentidão e erros para usuários; degradação do fluxo do e-commerce; aumento significativo do custo operacional.\n\n**Ação recomendada:** rollback de `dep-442` e posterior correção do fallback para apontar a um serviço distinto, com limite de profundidade seguro. O pedido de rollback foi aberto, mas **não exe

In [ ]:

# 1.    Incluir uma tool temporal pro agente
# 1.1.  Modificar o mockup de métricas. Criar um problema novo que exige investigação temporal
# 2.    Testar e avaliar se Agente funciona bem
# 3.    Portar para uma nova LLM (se for caso)

